# Browse agentcap parquets

Each row in an agentcap parquet is one captured `/v1/chat/completions`
call: the raw OpenAI request and response, plus a few provider-identity
columns. This notebook loads a bucket prefix as a dataset, picks a
row, and prints the conversation.

Replace `BUCKET` with your own `hf://buckets/<owner>/<name>/<prefix>/`.

In [ ]:
from datasets import load_dataset

BUCKET = "hf://buckets/dacorvo/agentcap-traces/transformers-coding-session/"
ds = load_dataset(BUCKET, split="train")
print(f"{len(ds)} rows")
ds.column_names

## Row shape

`request` and `response` are JSON strings (Arrow doesn't infer a
schema over heterogeneous tool-call shapes, so we hand it strings).
Everything else is flat.

In [ ]:
import json

row = ds[0]
req = json.loads(row["request"])
resp = json.loads(row["response"])

print(f"model         {row['model']}")
print(f"captured_at   {row['captured_at']}")
print(f"provider      {row.get('provider')}")
print(f"served_by     {row.get('served_by')}")
print(f"messages in   {len(req['messages'])}")
print(f"tools         {len(req.get('tools') or [])}")

## Print the conversation

OpenAI-spec messages: `role`, `content` (string or list of content-parts),
optional `tool_calls`. Streaming responses are captured as raw SSE bytes
under `response.raw` rather than a parsed body.

In [ ]:
def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") for p in content if p.get("type") == "text")
    return ""

for msg in req["messages"]:
    body = _text(msg.get("content"))
    print(f"--- {msg['role']} ---")
    print(body[:600] + ("…" if len(body) > 600 else ""))
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        print(f"  [tool_call] {fn.get('name')}({fn.get('arguments', '')[:200]})")
    print()

## Filter to one (agent, model) cell

A bucket prefix typically holds many `(agent, model)` tuples. The
filename embeds them; the rows themselves carry `model` and
`provider`. Combine with `served_by` to slice HF Router sub-provider
routing too.

In [ ]:
from collections import Counter

Counter((r["model"], r.get("provider"), r.get("served_by")) for r in ds)